# Hybrid Search — Files

Uses `hybrid_search_files` to combine semantic (vector) and keyword (FTS) search via RRF.

In [ ]:
import sys

sys.path.insert(0, '../src')

from sqlalchemy import text
from database import get_connection
from llm import embed_text

In [ ]:
# ── Search parameters ─────────────────────────────────────────────────────────
QUERY = 'indexing of source code for copilot retrieval'
PROJECT_ID = None  # set to an integer to filter by project, e.g. 1
MATCH_COUNT = 10
RRF_K = 60
# ──────────────────────────────────────────────────────────────────────────────

embedding = embed_text(QUERY)

with get_connection() as conn:
    rows = conn.execute(
        text(
            """
            SELECT id, project_id, path, summary, rrf_score
            FROM hybrid_search_files(
                :embedding,
                :query_text,
                :project_id,
                :match_count,
                :rrf_k
            )
            """
        ),
        {
            'embedding': str(embedding),
            'query_text': QUERY,
            'project_id': PROJECT_ID,
            'match_count': MATCH_COUNT,
            'rrf_k': RRF_K,
        },
    ).fetchall()

for rank, row in enumerate(rows, start=1):
    print(f'{rank:>2}. [{row.rrf_score:.4f}] (project {row.project_id}) {row.path}')
    if row.summary:
        print(f'    {row.summary[:120]}...')
    print()